In [1]:
from qiskit import *
from qiskit.quantum_info import Pauli, SparsePauliOp
from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit
from qiskit.circuit import ParameterVector

from VQE_and_QAOA import VQE_and_QAOA

import networkx as nx

from Function_gi_params import *


In [2]:
def Hamiltonian_qubo(N, edge_list, h_list, J_list):
    """Hamiltonian defined by a N vertex graph with connected edge in edge_list
    Args:
        N: number of qubits
        edge_list: list of edges(qubit index pairs)
        h_list: coefficients of single Pauli Z term
        J_list: coefficients of ZZ term
    Return:
        H: PauliSumOp, Hamiltonian

    """
    pauli_list = []
    for i in range(N):
        pauli_list.append(('Z', [i], h_list[i]))
        
    for k, (i, j) in enumerate(edge_list):
        pauli_list.append(('ZZ', [i, j], J_list[k]))
        
    H = SparsePauliOp.from_sparse_list(pauli_list, num_qubits = N)
    
    return H


def partition_graph(G):
    """
    Partition the edges of a given graph.
    G: Input graph (not necessarily complete)
    Returns: List of edge partitions
    """
    edges = list(G.edges())  # Get the edges of the graph
    n = G.number_of_nodes()  # Number of nodes in the graph
    pairs_all = []

    # Swapping indices for even and odd iterations
    swap_even = [i + pow(-1, i) for i in range(n)]
    swap_odd = [0]
    swap_odd.extend([i + pow(-1, i + 1) for i in range(1, n - 1)])
    swap_odd.append(n - 1)

    # Initial indices and first partition
    indexs = list(range(n))
    pairs_even = [(i, i + 1) for i in range(0, n, 2) if (i, i + 1) in edges or (i + 1, i) in edges]
    indexs = np.array(indexs)[swap_even]  # Apply initial swap
    pairs_all.append(pairs_even)

    # Iterate to create partitions
    for i in range(1, n):
        if i % 2 == 1:
            pair_odd = [(indexs[j], indexs[j + 1]) for j in range(1, n - 1, 2)
                        if (indexs[j], indexs[j + 1]) in edges or (indexs[j + 1], indexs[j]) in edges]
            pairs_all.append(pair_odd)
            indexs = np.array(indexs)[swap_odd]  # Swap for odd iteration
        else:
            pair_even = [(indexs[j], indexs[j + 1]) for j in range(0, n - 1, 2)
                         if (indexs[j], indexs[j + 1]) in edges or (indexs[j + 1], indexs[j]) in edges]
            pairs_all.append(pair_even)
            indexs = np.array(indexs)[swap_even]  # Swap for even iteration

    return pairs_all

In [13]:
n_qubits = 12
alpha = 1  #1, 0.01

ansatz_type =  'qaoa' ##'linear_cnot', 'parallel_cz'
layer = 2
tau = 0  ## tau only for warm start of structure-inspired ansatz
initialization = 'zeros' ## 'random' or 'zeros'
graph_density = 0


backend_method = 'matrix_product_state' ## 'statevector' or 'matrix_product_state'
shots = 1000
bond = 50


for r in range(1):
    gi_shots = 1000  ## shots for eatimating the expectation values of local pauli operators in warm start
    print('~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~')
    print('\nn_qubits: {}, \nr: {}, \nalpha: {}, \nshots: {}, \nansatz: {}, \nlayer: {}, \ntau: {}, \ninitialization: {}'\
        .format(n_qubits, r, alpha, shots, ansatz_type, layer, tau, initialization))

    ### folder to save result
    data_dir = './data/data_shots_{}_layer_{}/alpha_{}/N_{}/r_{}/ansatz_type_{}/'\
        .format(shots, layer, alpha, n_qubits, r, ansatz_type)
    os.makedirs(data_dir, exist_ok=True)

    #region load qubo instances, get Hamiltonian and edge_coeff_dict
    coeffs_file = f"../instances/{int(graph_density * 100):03}/N_{n_qubits}/QUBO_coeff_{n_qubits}V_r_{r}.txt"
    coeff_list = np.loadtxt(coeffs_file)

    graph_file = f"../instances/{int(graph_density * 100):03}/N_{n_qubits}/QUBO_{n_qubits}V_r_{r}.gpickle"
    with open(graph_file, 'rb') as f:
        G = pickle.load(f)
    edge_list = list(itertools.chain.from_iterable(partition_graph(G)))  ## used in bond dimension more than 100, same with Alice
    h_list = coeff_list[:n_qubits ]
    J_list = coeff_list[n_qubits : n_qubits + len(edge_list)]
    edge_coeff_dict = {}
    edge_coeff_dict.update({(i,): h_val for i, h_val in enumerate(h_list)})
    edge_coeff_dict.update({edge: J_val for edge, J_val in zip(edge_list, J_list)}) #CHANGED COMPARED TO THE OLD CODE
    
    H = Hamiltonian_qubo(n_qubits, edge_list, h_list, J_list)

    eigen_list = H.to_matrix(sparse=True).diagonal()
    #endregion

    ## order for two-qubit gate in circuit
    pairs_all = list(itertools.chain.from_iterable(partition(n_qubits)))

    #region get initial parameters
    if (ansatz_type) == 'structure_like_qubo_YZ_2':
        if initialization == 'warm_start_measure':
            gi_file_path = data_dir + '{}_tau_{}.pkl'.format(initialization, tau)
            layers_edge_params_dict, params_init, layers_exp_poss_dict = get_good_initial_params_measure(\
                n_qubits, tau, layer, edge_coeff_dict, pairs_all, eigen_list, shots, approximation, gi_file_path)
            print('\nwarm start fidelity', list(layers_exp_poss_dict['l_'+str(layer)].items())[0])
        elif initialization == 'warm_start_analy':
            gi_file_path = data_dir + '{}_tau_{}.pkl'.format(initialization, tau)
            edge_params_dict, params_init, layers_exp_poss_dict = get_good_initial_params_analy(\
                n_qubits, tau, layer, edge_coeff_dict, pairs_all, eigen_list, gi_file_path)
            print('\nwarm start fidelity', list(layers_exp_poss_dict['l_'+str(layer)].items())[0])
        elif initialization == 'zeros':
            params_init = np.zeros((n_qubits + 2*len(edge_list)) * layer)
        elif initialization == 'random':
            params_init = np.random.uniform(-np.pi, np.pi, (n_qubits + 2*len(edge_list)) * layer)
        else:
            raise ValueError('initialization method not found')
    elif (ansatz_type) == 'qaoa':
        if initialization == 'zeros':
            params_init = np.zeros(2 * layer) 
        elif initialization == 'random':
            params_init = np.random.uniform(-np.pi, np.pi, 2 * layer)
        else:
            raise ValueError('initialization method not found')
    else: # for efficient su2 ansatz, 
        if initialization == 'zeros':
            params_init = np.zeros(n_qubits * layer) 
        elif initialization == 'random':
            params_init = np.random.uniform(-np.pi, np.pi, n_qubits * layer)
        else:
            raise ValueError('initialization method not found')
    #endregion
    print('\ninitial parameters: ', params_init)



    if backend_method == 'matrix_product_state':
        backendoptions = {'method':backend_method, 'matrix_product_state_max_bond_dimension': bond, 'shots': shots}
    else:
        backendoptions = {'method':backend_method, 'shots': shots}


    vqe = VQE_and_QAOA(Hamiltonian = H, n_qubits = n_qubits, ansatz_type = ansatz_type, alpha = alpha, backendoptions=backendoptions,circuit_show = False, shots = shots)
    vqe.edge_coeff_dict = edge_coeff_dict

    E_min, E_max, ground_id_list = vqe.Get_minimun_from_H_mat()
    print('\nE_min from ED: ', E_min)
    for id in ground_id_list:
        print('ground state: ', np.binary_repr(id, n_qubits))
        Nnum = 0
        for s in np.binary_repr(id, n_qubits):
            Nnum += int(s)
        print('number of 1 in bitstring: ', Nnum)
    

    ## set the gate order in circuit
    print("\nStart optimization...")
    vqe.edge_list = pairs_all
    vqe.r_eval = []
    vqe.poss_eval= []
    vqe.cvar_eval = []
    vqe.std_eval = []
    final = minimize(vqe.CVaR_expectation,
                    params_init,
                    jac=False,
                    bounds=None,
                    method='COBYLA',
                    callback=None,
                    options={'maxiter': len(params_init)*20})

    save_list = np.array([vqe.cvar_eval, vqe.r_eval, vqe.poss_eval, vqe.std_eval])

    np.savetxt(data_dir + '/result_{}_tau_{}.txt'.format(initialization, tau), save_list.T)
    np.savetxt(data_dir + '/final_params_{}_tau_{}.txt'.format(initialization, tau), final.x)
    print('final fidelity: ', max(vqe.poss_eval))
    print('Write sucessfully to ' + data_dir)

    ### check the result
    print('\nvqe.cvar_eval[0]:', vqe.cvar_eval[0], '\nvqe.cvar_eval[-1]:', vqe.cvar_eval[-1])
    print("\nvqe.poss_eval[0]:", vqe.poss_eval[0], '\nvqe.poss_eval[-1]:', vqe.poss_eval[-1])


~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

n_qubits: 12, 
r: 0, 
alpha: 1, 
shots: 1000, 
ansatz: qaoa, 
layer: 2, 
tau: 0, 
initialization: zeros

initial parameters:  [0. 0. 0. 0.]

E_min from ED:  -11.3317
ground state:  111011011011
number of 1 in bitstring:  9

Start optimization...
final fidelity:  0.037
Write sucessfully to ./data/data_shots_1000_layer_2/alpha_1/N_12/r_0/ansatz_type_qaoa/

vqe.cvar_eval[0]: 0.09413959999998797 
vqe.cvar_eval[-1]: -5.99755020000001

vqe.poss_eval[0]: 0.001 
vqe.poss_eval[-1]: 0.028


(I-Zi)/2 * (I-Zi)/2